In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] ="2"

In [2]:
import torch
# 이후부터 torch는 물리 GPU 2만 “보임”
print(torch.cuda.device_count())  # 보통 1
print(torch.cuda.get_device_name(0))  # 물리 2번의 이름

1
NVIDIA GeForce RTX 3090


In [3]:
import numpy as np
import torch
from torch.utils.data import Dataset
import xarray as xr
import json
import os
from pathlib import Path
from typing import Tuple, Optional

MISSING_VALUE = -9.96921e36

class OISSTDataset(Dataset):
    """
    기존 OISSTDataset을 수정하여 meta 정보를 추가 반환
    """
    def __init__(
        self,
        ds,
        mask: np.ndarray,
        mean: float,
        std: float,
        tin: int = 14,
        tout: int = 7,
        var_name: str = "sst",
    ):
        self.tin = int(tin)
        self.tout = int(tout)

        self.mask = mask.astype(bool)  # (H, W)
        self.mean = float(mean)
        self.std = float(std) if float(std) != 0.0 else 1e-12

        arr = ds[var_name].values.astype(np.float32)  # (T, H, W)
        self.arr = arr
        self.time = ds.time.values

        self.T = arr.shape[0]
        self.H = arr.shape[1]
        self.W = arr.shape[2]

        if self.mask.shape != (self.H, self.W):
            raise ValueError(f"mask shape {self.mask.shape} != data spatial shape {(self.H, self.W)}")

        self.n = self.T - self.tin - self.tout + 1
        if self.n <= 0:
            raise ValueError("Not enough timesteps for the requested (tin, tout).")

    def __len__(self):
        return self.n

    def __getitem__(self, idx: int):
        seq = self.arr[idx : idx + self.tin + self.tout]  # (tin+tout, H, W)

        x = seq[: self.tin]
        y = seq[self.tin :]

        x = (x - self.mean) / self.std
        y = (y - self.mean) / self.std

        ocean = self.mask[None, :, :]
        x = np.where(ocean, x, 0.0)
        y = np.where(ocean, y, 0.0)

        x = torch.from_numpy(x).unsqueeze(1)  # (Tin, 1, H, W)
        y = torch.from_numpy(y).unsqueeze(1)  # (Tout, 1, H, W)

        # Meta 추가: month, day-of-year 등
        t_ref = self.time[idx + self.tin]  # np.datetime64
        t_py = np.datetime_as_string(t_ref, unit="D")  # 'YYYY-MM-DD'
        year, month, day = map(int, t_py.split("-"))

        # day-of-year (simple, no leap-year exactness issues? -> use numpy datetime arithmetic)
        jan1 = np.datetime64(f"{year}-01-01")
        doy = int((np.datetime64(t_py) - jan1).astype("timedelta64[D]").astype(int)) + 1

        meta = {
            "idx": int(idx),
            "t_ref": t_py,   # string date
            "year": int(year),
            "month": int(month),
            "doy": int(doy),
        }

        return x, y, meta


In [ ]:
def open_oisst(path: str | Path):
    path = str(path)
    if path.endswith(".zarr"):
        return xr.open_zarr(path)
    return xr.open_dataset(path)


def compute_mean_std_train(ds_train: xr.Dataset, mask: np.ndarray, var_name: str = "sst") -> Tuple[float, float]:
    arr = ds_train[var_name].values.astype(np.float32)  # (T, H, W)

    valid = np.isfinite(arr) & (arr != MISSING_VALUE)
    valid &= mask[None, :, :]

    x = arr[valid].astype(np.float64)
    
    mean = float(x.mean())
    std = float(x.std() + 1e-12)
    return mean, std


def build_datasets(
    data_path: str | Path,
    mask: np.ndarray,
    tin: int = 14,
    tout: int = 7,
    var_name: str = "sst",
    train_start: str = "1983-01-01",
    train_end: str = "2015-12-31",
    val_
    : str = "2016-01-01",
    val_end: str = "2020-12-31",

    stats_cache_json: Optional[str | Path] = None,
):
    mask = mask.astype(bool)

    ds = open_oisst(data_path)

    ds_train = ds.sel(time=slice(train_start, train_end))
    ds_val = ds.sel(time=slice(val_start, val_end))

    mean, std = compute_mean_std_train(ds_train, mask, var_name=var_name)

    if stats_cache_json is not None:
        p = Path(stats_cache_json)
        p.parent.mkdir(parents=True, exist_ok=True)
        with p.open("w", encoding="utf-8") as f:
            json.dump({"mean": mean, "std": std}, f, indent=2)

    val_set = OISSTDataset(ds_val, mask, mean, std, tin=tin, tout=tout, var_name=var_name)

    torch_mask = torch.from_numpy(mask).bool()  
    
    return val_set, torch_mask

In [ ]:
import os
import math
import random
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from losses import (
    masked_mse, masked_rmse, masked_mae, masked_mape, masked_r2,
    masked_rmse_per_t, masked_mae_per_t
)


def ensure_5d_batched(x: torch.Tensor) -> torch.Tensor:
    if x.dim() == 5:
        return x
    if x.dim() == 4:
        return x.unsqueeze(2)  # (B,T,1,H,W)
    raise ValueError(f"Expected 4D/5D batched tensor, got {tuple(x.shape)}")


def ensure_pred_5d(pred: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    if pred.dim() == 5:
        out = pred
    elif pred.dim() == 4:
        out = pred.unsqueeze(2)  # (B,T,1,H,W)
    else:
        raise ValueError(f"Expected pred 4D/5D, got {tuple(pred.shape)}")

    if out.shape[:2] != y.shape[:2] or out.shape[-2:] != y.shape[-2:]:
        raise ValueError(f"pred shape mismatch: pred={tuple(out.shape)} vs y={tuple(y.shape)}")
    if out.shape[2] != y.shape[2]:
        raise ValueError(f"channel mismatch: pred C={out.shape[2]} vs y C={y.shape[2]}")
    return out


def _extract_moe_probs_dict(model) -> dict:
    """
    모델 수정 없이 MoE head의 last_probs를 꺼내오기.
    return: {scale_key(str): probs(B,K)}  (가능한 것만)
    """
    net = getattr(model, "net", model)  # wrapper면 net, 아니면 model
    if not hasattr(net, "future_heads"):
        return {}

    probs_dict = {}
    for s, head in net.future_heads.items():
        p = getattr(head, "last_probs", None)
        if p is None:
            continue
        if not torch.is_tensor(p):
            continue
        probs_dict[str(s)] = p.detach()  # (B,K)
    return probs_dict


def _probs_stats(probs_dict: dict, eps: float = 1e-12) -> dict:
    """
    probs_dict: {scale: (B,K)}
    return:
      - moe_mean_probs: (K,)
      - moe_top1_share: (K,)
      - moe_entropy: float
      - moe_scale_mean_probs: {scale: (K,)}
    """
    if not probs_dict:
        return {}

    # scale별 평균
    scale_mean = {s: p.mean(dim=0) for s, p in probs_dict.items()}  # (K,)

    # 전체 평균/점유율은 scale*batch를 합쳐서
    all_p = torch.cat(list(probs_dict.values()), dim=0)  # (S*B, K)
    mean_p = all_p.mean(dim=0)                           # (K,)

    top1 = all_p.argmax(dim=-1)                          # (S*B,)
    top1_share = torch.bincount(top1, minlength=all_p.shape[-1]).float() / max(top1.numel(), 1)

    # entropy (평균 샘플 엔트로피)
    ent = -(all_p.clamp_min(eps) * all_p.clamp_min(eps).log()).sum(dim=-1).mean()

    return {
        "moe_mean_probs": mean_p.cpu().numpy(),
        "moe_top1_share": top1_share.cpu().numpy(),
        "moe_entropy": float(ent.item()),
        "moe_scale_mean_probs": {s: v.cpu().numpy() for s, v in scale_mean.items()},
    }


@torch.no_grad()
def eval_epoch(model, loader, mask_hw, device, eps_mape: float, collect_moe_stats: bool = True):
    model.eval()
    sums = {"mse": 0.0, "rmse": 0.0, "mae": 0.0, "mape": 0.0, "r2": 0.0}
    rmse_t_sum = None
    mae_t_sum = None
    preds = []
    n = 0

    # MoE 통계 누적용
    moe_all_probs = []              # list of (S*B,K) chunk
    moe_scale_probs_accum = {}      # scale -> list of (B,K)

    for x, y, meta in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        x = ensure_5d_batched(x)
        y = ensure_5d_batched(y)

        pred = model(x)
        pred = ensure_pred_5d(pred, y)
        preds.append(pred.detach().cpu())

        sums["mse"] += masked_mse(pred, y, mask_hw).item()
        sums["rmse"] += masked_rmse(pred, y, mask_hw).item()
        sums["mae"] += masked_mae(pred, y, mask_hw).item()
        sums["mape"] += masked_mape(pred, y, mask_hw, eps=eps_mape, as_percent=True).item()
        sums["r2"] += masked_r2(pred, y, mask_hw).item()

        rmse_t = masked_rmse_per_t(pred, y, mask_hw).detach().cpu()
        mae_t = masked_mae_per_t(pred, y, mask_hw).detach().cpu()

        if rmse_t_sum is None:
            rmse_t_sum = rmse_t.clone()
            mae_t_sum = mae_t.clone()
        else:
            rmse_t_sum += rmse_t
            mae_t_sum += mae_t

        # ---- MoE probs 수집 (forward 직후 head.last_probs에 들어있음) ----
        if collect_moe_stats:
            probs_dict = _extract_moe_probs_dict(model)
            if probs_dict:
                # 전체용
                moe_all_probs.append(torch.cat(list(probs_dict.values()), dim=0).cpu())  # (S*B,K)

                # scale별 누적
                for s, p in probs_dict.items():
                    moe_scale_probs_accum.setdefault(s, []).append(p.cpu())

        n += 1

    for k in sums:
        sums[k] /= max(n, 1)

    if rmse_t_sum is None:
        sums["rmse_t"] = None
        sums["mae_t"] = None
    else:
        sums["rmse_t"] = (rmse_t_sum / max(n, 1)).tolist()
        sums["mae_t"] = (mae_t_sum / max(n, 1)).tolist()

    # ---- MoE stats finalize ----
    if collect_moe_stats and len(moe_all_probs) > 0:
        all_p = torch.cat(moe_all_probs, dim=0)  # (total_samples_across_batches, K)
        mean_p = all_p.mean(dim=0)
        top1 = all_p.argmax(dim=-1)
        top1_share = torch.bincount(top1, minlength=all_p.shape[-1]).float() / max(top1.numel(), 1)
        ent = -(all_p.clamp_min(1e-12) * all_p.clamp_min(1e-12).log()).sum(dim=-1).mean()

        sums["moe_mean_probs"] = mean_p.numpy()
        sums["moe_top1_share"] = top1_share.numpy()
        sums["moe_entropy"] = float(ent.item())

        # scale별 평균
        scale_mean = {}
        for s, plist in moe_scale_probs_accum.items():
            ps = torch.cat(plist, dim=0)   # (total_B, K)
            scale_mean[s] = ps.mean(dim=0).numpy()
        sums["moe_scale_mean_probs"] = scale_mean
    else:
        sums["moe_mean_probs"] = None
        sums["moe_top1_share"] = None
        sums["moe_entropy"] = None
        sums["moe_scale_mean_probs"] = None

    return sums, preds


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
def spatial_error_map(pred: torch.Tensor, target: torch.Tensor, mask_hw: torch.Tensor) -> np.ndarray:
    """
    Args:
        pred: (B, T, 1, H, W)
        target: (B, T, 1, H, W)
    Returns:
        np.ndarray: (H, W) 모양의 평균 에러 맵
    """

    error_map = torch.abs(pred - target) # (B, T, 1, H, W)
    

    error_map = error_map.squeeze(2).mean(dim=1) # (B, H, W)
    error_map = error_map.mean(dim=0) # (H, W)
    

    error_map = error_map * mask_hw
    return error_map.cpu().numpy()

def plot_spatial_error_map(error_map: np.ndarray, title: str = "Spatial Error Map"):
    plt.figure(figsize=(10, 6))
    

    error_map_masked = np.where(error_map == 0, np.nan, error_map)
    
    plt.imshow(error_map_masked, cmap='YlOrRd', interpolation='nearest')
    plt.colorbar(label='Mean Absolute Error')
    plt.title(title)
    plt.axis('off')
    plt.show()

In [ ]:
def horizon_curve_analysis(pred: torch.Tensor, target: torch.Tensor, mask_hw: torch.Tensor) -> np.ndarray:
    rmse_t = []
    for t in range(pred.shape[1]):
        p_t = pred[:, t:t+1]
        y_t = target[:, t:t+1]
        
        # .detach()를 추가하여 연산 그래프에서 분리합니다.
        rmse_val = masked_rmse(p_t, y_t, mask_hw).detach().cpu().numpy()
        rmse_t.append(rmse_val)
        
    return np.array(rmse_t)

def plot_horizon_curve(rmse_t: np.ndarray, title: str = "Horizon Curve (RMSE)"):
    """
    Visualize the RMSE over the time horizon (t=1..7).
    Args:
        rmse_t (np.ndarray): RMSE values for each time step (1 to 7)
        title (str): The title for the plot
    """
    plt.figure(figsize=(8, 6))
    plt.plot(np.arange(1, len(rmse_t) + 1), rmse_t, marker='o')
    plt.title(title)
    plt.xlabel("Time step (t=1..7)")
    plt.ylabel("RMSE")
    plt.grid(True)
    plt.show()


In [ ]:
def season_bucket_analysis(month: np.ndarray) -> np.ndarray:
    """
    Assign each sample to a season bucket based on the month.
    Args:
        month (np.ndarray): Array of months (1 to 12)
    
    Returns:
        np.ndarray: Season bucket (0=DJF, 1=MAM, 2=JJA, 3=SON)
    """
    season = np.zeros_like(month)
    season[(month == 12) | (month == 1) | (month == 2)] = 0  # DJF
    season[(month == 3) | (month == 4) | (month == 5)] = 1  # MAM
    season[(month == 6) | (month == 7) | (month == 8)] = 2  # JJA
    season[(month == 9) | (month == 10) | (month == 11)] = 3  # SON
    return season

def plot_seasonal_performance(df: pd.DataFrame, title: str = "Seasonal RMSE"):
    seasons_labels = ['DJF', 'MAM', 'JJA', 'SON']
    
    # 계절(0,1,2,3)별로 그룹화하여 평균 RMSE 계산
    # 만약 데이터에 특정 계절이 없다면 NaN이 나올 수 있으므로 reindex로 고정
    rmse_season = df.groupby('season')['rmse_total'].mean().reindex([0, 1, 2, 3])
    
    plt.figure(figsize=(8, 6))
    plt.bar(seasons_labels, rmse_season, color=['skyblue', 'lightgreen', 'orange', 'salmon'])
    plt.title(title)
    plt.xlabel("Season")
    plt.ylabel("Average RMSE")
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

In [ ]:
import os
import numpy as np
import torch
import pandas as pd

@torch.no_grad()
def run_spatial_temporal_analysis(model, val_loader, mask_hw, device, model_name="MSFv2", title="Model Analysis"):
    """
    모델의 성능을 분석하고 예측값을 .npy 파일로 저장합니다.
    """
    model.eval()
    all_preds = []
    all_targets = []
    all_meta_months = []
    sample_rmse_list = []

    print(f"Starting Analysis and Saving for {model_name}...")
    
    mask_cpu = mask_hw.cpu()

    for x, y, meta in val_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        
        p_cpu = pred.detach().cpu()
        y_cpu = y.detach().cpu()
        
        all_preds.append(p_cpu)
        all_targets.append(y_cpu)
        
        # 월 정보 수집
        m = meta['month']
        all_meta_months.extend(m.cpu().numpy().tolist() if torch.is_tensor(m) else list(m))
            
        # 샘플별/타임스텝별 RMSE 계산 (계절별 분석용)
        for b in range(p_cpu.shape[0]):
            t_rmses = [masked_rmse(p_cpu[b, t], y_cpu[b, t], mask_cpu).item() for t in range(p_cpu.shape[1])]
            sample_rmse_list.append(t_rmses)

    # 전체 데이터 결합 및 넘파이 변환 (N, T, C, H, W)
    final_preds = torch.cat(all_preds, dim=0).numpy()
    final_targets = torch.cat(all_targets, dim=0).numpy()
    sample_rmse_array = np.array(sample_rmse_list)

    # --- [핵심] 모델별 .npy 파일 저장 로직 ---
    save_dir = "model_comparison"
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    # 모델별 예측값 저장 (예: model_comparison/MSFv2_preds.npy)
    pred_save_path = os.path.join(save_dir, f"{model_name}_preds.npy")
    np.save(pred_save_path, final_preds)
    
    # 정답지는 공통이므로 파일이 없을 때만 한 번 저장
    gt_save_path = os.path.join(save_dir, "ground_truth.npy")
    np.save(gt_save_path, final_targets)

    print(f"Successfully saved {model_name} predictions to {pred_save_path}")

    # --- 기존 시각화 로직 ---
    # 1. Spatial Error Map
    spatial_errors = spatial_error_map(torch.from_numpy(final_preds), torch.from_numpy(final_targets), mask_cpu)
    if spatial_errors.ndim == 3: spatial_errors = spatial_errors.mean(axis=0)
    plot_spatial_error_map(spatial_errors, title=f"{model_name} - Spatial Error")

    # 2. Horizon Curve
    avg_rmse_t = sample_rmse_array.mean(axis=0)
    plot_horizon_curve(avg_rmse_t, title=f"{model_name} - Horizon Curve")

    # 3. Seasonal Performance
    season_buckets = season_bucket_analysis(np.array(all_meta_months))
    df = pd.DataFrame({
        'season': season_buckets,
        'rmse_total': sample_rmse_array.mean(axis=1)
    })
    plot_seasonal_performance(df, title=f"{model_name} - Seasonal RMSE")
    
    return final_preds

In [ ]:
def result(model, cfg: dict, ckpt_path: str, model_name: str):
    set_seed(int(cfg.get("seed", 42)))

    mask = np.load(cfg["mask_path"]).astype(bool)
    H, W = mask.shape
    cfg["image_size"] = int(H)  # keep consistent with dataset grid

    val_set, mask_hw = build_datasets(
    data_path=cfg["data_path"],
    mask=mask,
    tin=cfg["input_len"],
    tout=cfg["pred_len"],
    var_name=cfg["var_name"],
    train_start=cfg["train_start"],
    train_end=cfg["train_end"],
    val_start=cfg["test_start"],
    val_end=cfg["test_end"],
    stats_cache_json=cfg["stats_cache"],
)

    use_cuda = torch.cuda.is_available() and str(cfg.get("device", "cuda")).startswith("cuda")
    device = torch.device("cuda" if use_cuda else "cpu")

    bs = int(cfg.get("batch_size", 64))
    nw = int(cfg.get("num_workers", 4))
    eps_mape = float(cfg.get("eps_mape", 1e-3))

    pin = True if use_cuda else False

    val_loader = DataLoader(
        val_set, batch_size=bs, shuffle=False, num_workers=nw,
        pin_memory=pin, persistent_workers=(nw > 0), drop_last=True
    )

    model = model.to(device)
    mask_hw = mask_hw.to(device)


    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"])

    val_sums, pred = eval_epoch(model, val_loader, mask_hw, device, eps_mape)

    print("moe_mean_probs:", val_sums["moe_mean_probs"])
    print("moe_top1_share:", val_sums["moe_top1_share"])
    print("moe_entropy:", val_sums["moe_entropy"])
    print("moe_scale_mean_probs:", val_sums["moe_scale_mean_probs"])
    
    run_spatial_temporal_analysis(model, val_loader, mask_hw, device, model_name)

    print(
        f"VAL | MSE {val_sums['mse']:.6f} RMSE {val_sums['rmse']:.6f} MAE {val_sums['mae']:.6f} "
        f"MAPE {val_sums['mape']:.3f} R2 {val_sums['r2']:.4f}"
    )

    return val_sums, pred

In [ ]:
# import os
# import csv
# import numpy as np
# import torch

# from configs import get_default_cfg
# from model.models.msf3_model_scalehead_moe import MSFv3_ScaleHeadMoE, MSFGv3_ScaleHeadMoE

# cfg = get_default_cfg()

# device = cfg["device"]
# in_shape = (cfg["input_len"], 1, 32, 32)

# model = MSFv3_ScaleHeadMoE(
# image_size=cfg["image_size"],
# in_chans=1,
# tin=cfg["input_len"],
# tout=cfg["pred_len"],
# d_model=cfg["d_model"],
# depth=cfg["depth"],
# num_heads=cfg["num_heads"],
# mlp_ratio=cfg["mlp_ratio"],
# dropout=cfg["dropout"],
# attn_dropout=cfg["attn_dropout"],
# spatial_scales=cfg["spatial_scales"],
# fusion=cfg["fusion"],
# time_pool=cfg["time_pool"],

# head_num_experts=cfg["head_num_experts"],
# head_top_k=cfg["head_top_k"],
# head_temperature=cfg["head_temperature"],
# aux_balance_coef=cfg["aux_balance_coef"],
# aux_z_coef=cfg["aux_z_coef"],
# ).to(device)


    
# ckpt_path = "checkpoints/msfv2_moe4_base00.pt"

# val_results, predictions = result(
#     model=model, 
#     cfg=cfg, 
#     ckpt_path=ckpt_path,
#     model_name="MSFv3_scalehead_moe4_base00"
# )

In [ ]:
# import os
# import csv
# import numpy as np
# import torch

# from configs import get_default_cfg
# from model.models.msf3_model import MSFv3, MSFGv3, MSFG2v3



# cfg = get_default_cfg()

# device = cfg["device"]

# model = MSFv3(
#     image_size=cfg["image_size"],
#     in_chans=1,
#     tin=cfg["input_len"],
#     tout=cfg["pred_len"],
#     d_model=cfg["d_model"],
#     depth=cfg["depth"],
#     num_heads=cfg["num_heads"],
#     mlp_ratio=cfg["mlp_ratio"],
#     dropout=cfg["dropout"],
#     attn_dropout=cfg["attn_dropout"],
#     spatial_scales=cfg["spatial_scales"],
#     fusion="sum",
#     time_pool="last"
# ).to(device)

# ckpt_path = "checkpoints/msfv3_base03.pt"

# val_results, predictions = result(
#     model=model, 
#     cfg=cfg, 
#     ckpt_path=ckpt_path,
#     model_name="msfv3_base03"
# )


In [ ]:
# import os
# import csv
# import copy
# import numpy as np
# import torch

# from configs import get_default_cfg
# from model.models.msf3_model_encoding import MSFv3, MSFGv3, MSFG2v3

# cfg = get_default_cfg()
# device = cfg["device"]


# # 1. 초기 모델
# model_init = MSFv3(
#     image_size=cfg["image_size"],
#     in_chans=1,
#     tin=cfg["input_len"],
#     tout=cfg["pred_len"],
#     d_model=cfg["d_model"],
#     depth=cfg["depth"],
#     num_heads=cfg["num_heads"],
#     mlp_ratio=cfg["mlp_ratio"],
#     dropout=cfg["dropout"],
#     attn_dropout=cfg["attn_dropout"],
#     spatial_scales=cfg["spatial_scales"],
#     fusion="sum",
#     time_pool="last",
#     use_time_emb=True,
#     use_scale_emb=True,
#     freeze_time_emb=True,
#     freeze_scale_emb=True
#     # use_time_emb= False,
#     # use_scale_emb=False,
#     # freeze_time_emb=False,
#     # freeze_scale_emb=False
# ).to(device)

# # 2. 학습 후 weight를 올릴 별도 모델
# model_trained = MSFv3(
#     image_size=cfg["image_size"],
#     in_chans=1,
#     tin=cfg["input_len"],
#     tout=cfg["pred_len"],
#     d_model=cfg["d_model"],
#     depth=cfg["depth"],
#     num_heads=cfg["num_heads"],
#     mlp_ratio=cfg["mlp_ratio"],
#     dropout=cfg["dropout"],
#     attn_dropout=cfg["attn_dropout"],
#     spatial_scales=cfg["spatial_scales"],
#     fusion="sum",
#     time_pool="last",
#     use_time_emb=True,
#     use_scale_emb=True,
#     freeze_time_emb=True,
#     freeze_scale_emb=True
# ).to(device)

# # ckpt_path = "checkpoints/msfv3_ec_x.pt"
# # val_results, predictions = result(
# #     model=model_init, 
# #     cfg=cfg, 
# #     ckpt_path=ckpt_path,
# #     model_name="msfv3_ec_x"
# # )

# ckpt_path = "checkpoints/msfv3_ec_f.pt"
# val_results, predictions = result(
#     model=model_trained, 
#     cfg=cfg, 
#     ckpt_path=ckpt_path,
#     model_name="msfv3_ec_f"
# )

# net_init = model_init.net if hasattr(model_init, "net") else model_init
# net_trained = model_trained.net if hasattr(model_trained, "net") else model_trained

In [ ]:
# print("=== trained time_emb ===")
# print(net_trained.time_emb.weight[:3, :8])

# print("=== init scale_emb ===")
# print(net_init.scale_emb.weight[:, :8])



MSF_base

=== trained time_emb ===
tensor([[-0.7687, -1.0270, -0.4790,  0.6748,  0.3810, -2.1215,  1.1566, -0.5128],
        [ 1.3277,  0.1642,  1.0510, -1.3180,  2.6631,  0.2445, -0.6709,  0.3228],
        [-0.8087, -1.4812,  0.6260, -1.0652,  0.0261, -1.9474, -0.0239,  1.5109]],
       device='cuda:0', grad_fn=<SliceBackward0>)

=== trained scale_emb ===
tensor([[-0.4474, -0.5407,  0.6096, -0.1459, -0.6400, -0.0710, -1.3490, -0.0690],
        [ 1.2054,  0.5998,  0.3868, -0.9071,  1.0530,  0.2106,  0.3231, -0.7284],
        [-1.2817, -0.2707,  0.3871, -0.5635,  0.7768,  0.0280,  0.4479, -1.1295],
        [ 0.1583, -0.5593,  1.5042,  0.4330,  1.0713, -0.2485, -0.3429,  0.3625]],
       device='cuda:0', grad_fn=<SliceBackward0>)

MSF_Freeze

=== trained time_emb ===
tensor([[-0.7635, -1.0258, -0.4862,  0.6853,  0.3704, -2.1299,  1.1543, -0.5239],
        [ 1.3257,  0.1682,  1.0538, -1.3082,  2.6663,  0.2377, -0.6800,  0.3239],
        [-0.8004, -1.4799,  0.6224, -1.0646,  0.0240, -1.9548, -0.0401,  1.5147]],
       device='cuda:0')
       
=== trained scale_emb ===
tensor([[-0.4480, -0.5421,  0.6115, -0.1470, -0.6382, -0.0703, -1.3540, -0.0675],
        [ 1.2063,  0.5998,  0.3872, -0.9116,  1.0527,  0.2063,  0.3228, -0.7303],
        [-1.2829, -0.2617,  0.3807, -0.5629,  0.7793,  0.0208,  0.4478, -1.1119],
        [ 0.1618, -0.5752,  1.5090,  0.4284,  1.0707, -0.2498, -0.3381,  0.3639]],
       device='cuda:0')

In [ ]:
# net_init = model_init.net if hasattr(model_init, "net") else model_init
# net_trained = model_trained.net if hasattr(model_trained, "net") else model_trained

# time_diff = (net_trained.time_emb.weight - net_init.time_emb.weight).detach().cpu()
# scale_diff = (net_trained.scale_emb.weight - net_init.scale_emb.weight).detach().cpu()

# print("time_emb change norm      :", time_diff.norm().item())
# print("time_emb mean abs change  :", time_diff.abs().mean().item())
# print("scale_emb change norm     :", scale_diff.norm().item())
# print("scale_emb mean abs change :", scale_diff.abs().mean().item())

In [ ]:
import os
import csv
import numpy as np
import torch

from configs import get_default_cfg
from model.models.swinlstm_model import SwinLSTM_B_Model

cfg = get_default_cfg()
device = cfg["device"]

model = SwinLSTM_B_Model(
    configs= cfg
).to(cfg["device"])


ckpt_path = "checkpoints/swinLstm_B.pt"

val_results, predictions = result(
    model=model, 
    cfg=cfg, 
    ckpt_path=ckpt_path,
    model_name="swinLstm_B"
)

KeyError: 'in_shape'

In [ ]:
# import os
# import csv
# import numpy as np
# import torch

# from configs import get_default_cfg
# from model.models.msf3_model import MSFv3, MSFGv3, MSFG2v3
# from model.models.msf3_resdecode_model import MSFv3ResDecode, MSFGv3ResDecode, MSFG2v3ResDecode


# cfg = get_default_cfg()

# device = cfg["device"]

# ModelCls = (
#         MSFv3ResDecode if cfg["fusion"] == "sum" else
#         MSFGv3ResDecode if cfg["fusion"] == "gated" else
#         MSFG2v3ResDecode
#     )

# model = ModelCls(
#         image_size=cfg["image_size"],
#         in_chans=1,
#         tin=cfg["input_len"],
#         tout=cfg["pred_len"],
#         d_model=cfg["d_model"],
#         depth=cfg["depth"],
#         num_heads=cfg["num_heads"],
#         mlp_ratio=cfg["mlp_ratio"],
#         dropout=cfg["dropout"],
#         attn_dropout=cfg["attn_dropout"],
#         spatial_scales=cfg["spatial_scales"],
#         fusion=cfg["fusion"],
#         use_pos_emb=cfg["use_pos_emb"],
#         time_pool="last",
#     ).to(device)


# ckpt_path = "checkpoints/msfv3_resdecode_base03.pt"

# val_results, predictions = result(
#     model=model, 
#     cfg=cfg, 
#     ckpt_path=ckpt_path,
#     model_name="msfv3_resdecode_base03"
# )

In [ ]:
# import os
# import csv
# import numpy as np
# import torch

# from configs import get_default_cfg
# from model.models.sst_simvp_model import SimVP_Model


# cfg = get_default_cfg()

# device = cfg["device"]
# in_shape = (cfg["input_len"], 1, 32, 32)

# model = SimVP_Model(
#     in_shape=in_shape,
#     t_out=cfg["pred_len"],
#     hid_S=cfg["hid_S"],
#     hid_T=cfg["hid_T"],
#     N_S=cfg["N_S"],
#     N_T=cfg["N_T"],
#     model_type="gsta",
# ).to(device)

# ckpt_path = "checkpoints/simvp_gsta.pt"

# val_results, predictions = result(
#     model=model, 
#     cfg=cfg, 
#     ckpt_path=ckpt_path,
#     model_name="simvp_gsta"
# )

In [ ]:
# import os
# import csv
# import numpy as np
# import torch

# from configs import get_default_cfg
# from model.models.sst_simvp_model import SimVP_Model


# cfg = get_default_cfg()

# device = cfg["device"]
# in_shape = (cfg["input_len"], 1, 32, 32)

# model = SimVP_Model(
#     in_shape=in_shape,
#     t_out=cfg["pred_len"],
#     hid_S=cfg["hid_S"],
#     hid_T=cfg["hid_T"],
#     N_S=cfg["N_S"],
#     N_T=cfg["N_T"],
#     model_type="incepu",
# ).to(device)

# ckpt_path = "checkpoints/simvp_incepu.pt"

# val_results, predictions = result(
#     model=model, 
#     cfg=cfg, 
#     ckpt_path=ckpt_path,
#     model_name="SimVP_INCEPU"
# )

In [ ]:
# import os
# import csv
# import numpy as np
# import torch

# from configs import get_default_cfg
# from model.models.convlstm_model import ConvLSTM_Model


# cfg = get_default_cfg()

# device = cfg["device"]
# in_shape = (cfg["input_len"], 1, 32, 32)

# model = ConvLSTM_Model(
#     num_layers=cfg["num_layers"],
#     num_hidden=cfg["num_hidden"],
#     configs=cfg,
# ).to(cfg["device"])

# ckpt_path = "checkpoints/convlstm.pt"

# val_results, predictions = result(
#     model=model, 
#     cfg=cfg, 
#     ckpt_path=ckpt_path,
#     model_name="ConvLSTM"
# )

In [ ]:
# import os
# import csv
# import numpy as np
# import torch

# from configs import get_default_cfg
# from model.models.predrnn_model import PredRNN_Model


# cfg = get_default_cfg()

# device = cfg["device"]
# in_shape = (cfg["input_len"], 1, 32, 32)

# model = PredRNN_Model(
#     num_layers=int(cfg["num_layers"]),
#     num_hidden=list(cfg["num_hidden"]),
#     configs=cfg,
# ).to(cfg["device"])
    
# ckpt_path = "checkpoints/predrnn.pt"

# val_results, predictions = result(
#     model=model, 
#     cfg=cfg, 
#     ckpt_path=ckpt_path,
#     model_name="predrnn"
# )

In [ ]:
# import os
# import csv
# import numpy as np
# import torch

# from configs import get_default_cfg
# from model.models.mim_model import MIM_Model


# cfg = get_default_cfg()

# device = cfg["device"]
# in_shape = (cfg["input_len"], 1, 32, 32)

# model = MIM_Model(
#     num_layers=int(cfg["num_layers"]),
#     num_hidden=list(cfg["num_hidden"]),
#     configs=cfg,
# ).to(cfg["device"])
    
# ckpt_path = "checkpoints/MIM.pt"

# val_results, predictions = result(
    
#     model=model, 
#     cfg=cfg, 
#     ckpt_path=ckpt_path,
#     model_name="MIM"
# )

In [ ]:
# import os
# import csv
# import numpy as np
# import torch

# from configs import get_default_cfg
# from model.models.mau_model import MAU_Model


# cfg = get_default_cfg()

# device = cfg["device"]
# in_shape = (cfg["input_len"], 1, 32, 32)

# model = MAU_Model(
#     configs=cfg,
# ).to(cfg["device"])
    
# ckpt_path = "checkpoints/MAU.pt"

# val_results, predictions = result(
#     model=model, 
#     cfg=cfg, 
#     ckpt_path=ckpt_path,
#     model_name="MAU"
# )

In [ ]:
# import os
# import csv
# import numpy as np
# import torch

# from configs import get_default_cfg
# from model.models.vit_gru_model import TorchVisionViTGRUForecast

# cfg = get_default_cfg()

# device = cfg["device"]
# in_shape = (cfg["input_len"], 1, 32, 32)

# model = TorchVisionViTGRUForecast(
#     image_size=cfg["image_size"],     # should match H/W if fixed
#     patch_size=cfg["patch_size"],
#     in_chans=1,
#     tin=cfg["input_len"],
#     tout=cfg["pred_len"],
#     embed_dim=cfg["embed_dim"],
#     depth=cfg["depth"],
#     num_heads=cfg["num_heads"],
#     mlp_ratio=cfg["mlp_ratio"],
#     dropout=cfg["vit_dropout"],
#     attn_dropout=cfg["vit_attn_dropout"],
#     freeze_vit=cfg["vit_freeze"],
# ).to(device)
    
# ckpt_path = "checkpoints/vit_gru.pt"

# val_results, predictions = result(
#     model=model, 
#     cfg=cfg, 
#     ckpt_path=ckpt_path,
#     model_name="ViT_GRU"
# )

In [ ]:
# import os
# import csv
# import numpy as np
# import torch

# from configs import get_default_cfg
# from model.models.TimeSformer_model import TimeSformer


# cfg = get_default_cfg()

# device = cfg["device"]
# in_shape = (cfg["input_len"], 1, 32, 32)

# model = TimeSformer(
#     image_size=cfg["image_size"],
#     spatial_patch=cfg["spatial_patch"],
#     in_chans=1,
#     tin=cfg["input_len"],
#     tout=cfg["pred_len"],
#     d_model=cfg["d_model"],
#     depth=cfg["depth"],
#     num_heads=cfg["num_heads"],
#     mlp_ratio=cfg["mlp_ratio"],
#     dropout=cfg["dropout"],
# )
    
# ckpt_path = "checkpoints/TimeSformer.pt"

# val_results, predictions = result(
#     model=model, 
#     cfg=cfg, 
#     ckpt_path=ckpt_path,
#     model_name="TimeSformer"
# )

In [ ]:
# import os
# import csv
# import numpy as np
# import torch

# from configs import get_default_cfg
# from model.models.TimeSformer_model_2 import TimeSformer


# cfg = get_default_cfg()

# device = cfg["device"]
# in_shape = (cfg["input_len"], 1, 32, 32)

# model = TimeSformer(
#     image_size=cfg["image_size"],
#     spatial_patch=cfg["spatial_patch"],
#     in_chans=1,
#     tin=cfg["input_len"],
#     tout=cfg["pred_len"],
#     d_model=cfg["d_model"],
#     depth=cfg["depth"],
#     num_heads=cfg["num_heads"],
#     mlp_ratio=cfg["mlp_ratio"],
#     dropout=cfg["dropout"],
# )
    
# ckpt_path = "checkpoints/TimeSformer2.pt"

# val_results, predictions = result(
#     model=model, 
#     cfg=cfg, 
#     ckpt_path=ckpt_path,
#     model_name="TimeSformer2"
# )

In [ ]:
# import os
# import csv
# import numpy as np
# import torch

# from configs import get_default_cfg
# from model.models.msf2_model import MSFv2, MSFGv2


# cfg = get_default_cfg()

# device = cfg["device"]
# in_shape = (cfg["input_len"], 1, 32, 32)

# model = MSFv2(
#         image_size=cfg["image_size"],
#         in_chans=1,
#         tin=cfg["input_len"],
#         tout=cfg["pred_len"],
#         d_model=cfg["d_model"],
#         depth=cfg["depth"],
#         num_heads=cfg["num_heads"],
#         mlp_ratio=cfg["mlp_ratio"],
#         dropout=cfg["dropout"],
#         attn_dropout=cfg["attn_dropout"],
#         spatial_scales=cfg["spatial_scales"],
#         fusion=cfg["fusion"],          # v2에서 사용
#         use_pad=cfg["use_pad"],
#         use_pos_emb=cfg["use_pos_emb"],
#         time_pool=cfg["time_pool"],    # v2가 지원한다는 가정
#     ).to(device)
    
# ckpt_path = "checkpoints/msfv2.pt"

# val_results, predictions = result(
#     model=model, 
#     cfg=cfg, 
#     ckpt_path=ckpt_path,
#     model_name="MSFv2"
# )

In [ ]:
# import os
# import csv
# import numpy as np
# import torch

# from configs import get_default_cfg
# from model.models.msf2_model import MSFv2, MSFGv2


# cfg = get_default_cfg()

# device = cfg["device"]
# in_shape = (cfg["input_len"], 1, 32, 32)

# model = MSFv2(
#         image_size=cfg["image_size"],
#         in_chans=1,
#         tin=cfg["input_len"],
#         tout=cfg["pred_len"],
#         d_model=cfg["d_model"],
#         depth=cfg["depth"],
#         num_heads=cfg["num_heads"],
#         mlp_ratio=cfg["mlp_ratio"],
#         dropout=cfg["dropout"],
#         attn_dropout=cfg["attn_dropout"],
#         spatial_scales=cfg["spatial_scales"],
#         fusion=cfg["fusion"],          # v2에서 사용
#         use_pad=cfg["use_pad"],
#         use_pos_emb=cfg["use_pos_emb"],
#         time_pool=cfg["time_pool"],    # v2가 지원한다는 가정
#     ).to(device)
    
# ckpt_path = "checkpoints/msfv2_24816.pt"

# val_results, predictions = result(
#     model=model, 
#     cfg=cfg, 
#     ckpt_path=ckpt_path,
#     model_name="MSFv2_24816"
# )

In [ ]:
# import os
# import csv
# import numpy as np
# import torch

# from configs import get_default_cfg
# from model.models.msf2_model_gate import MSFv2


# cfg = get_default_cfg()

# device = cfg["device"]
# in_shape = (cfg["input_len"], 1, 32, 32)

# model = MSFv2(
#     image_size=cfg["image_size"],
#     in_chans=1,
#     tin=cfg["input_len"],
#     tout=cfg["pred_len"],
#     d_model=cfg["d_model"],
#     depth=cfg["depth"],
#     num_heads=cfg["num_heads"],
#     mlp_ratio=cfg["mlp_ratio"],
#     dropout=cfg["dropout"],
#     attn_dropout=cfg["attn_dropout"],
#     spatial_scales=cfg["spatial_scales"],
#     use_pad=cfg["use_pad"],
#     use_pos_emb=cfg["use_pos_emb"],
#     time_pool=cfg["time_pool"],

#     # MoM fusion params
#     mom_top_k=cfg["mom_top_k"],
#     mom_temperature=cfg["mom_temperature"],
#     aux_balance_coef=cfg["aux_balance_coef"],
#     aux_z_coef=cfg["aux_z_coef"],
# )

    
# ckpt_path = "checkpoints/msfv2_gate.pt"

# val_results, predictions = result(
#     model=model, 
#     cfg=cfg, 
#     ckpt_path=ckpt_path,
#     model_name="MSFv2_gate"
# )

In [ ]:
# import os
# import csv
# import numpy as np
# import torch

# from configs import get_default_cfg
# from model.models.msf2_model_scalehead_moe import MSFv2, MSFGv2

# cfg = get_default_cfg()

# device = cfg["device"]
# in_shape = (cfg["input_len"], 1, 32, 32)

# model = MSFv2(
# image_size=cfg["image_size"],
# in_chans=1,
# tin=cfg["input_len"],
# tout=cfg["pred_len"],
# d_model=cfg["d_model"],
# depth=cfg["depth"],
# num_heads=cfg["num_heads"],
# mlp_ratio=cfg["mlp_ratio"],
# dropout=cfg["dropout"],
# attn_dropout=cfg["attn_dropout"],
# spatial_scales=cfg["spatial_scales"],
# fusion=cfg["fusion"],
# use_pad=cfg["use_pad"],
# use_pos_emb=cfg["use_pos_emb"],
# time_pool=cfg["time_pool"],

# head_num_experts=cfg["head_num_experts"],
# head_top_k=cfg["head_top_k"],
# head_temperature=cfg["head_temperature"],
# aux_balance_coef=cfg["aux_balance_coef"],
# aux_z_coef=cfg["aux_z_coef"],
# ).to(device)


    
# ckpt_path = "checkpoints/msfv2_scalehead_moe_k4_soft_sum2.pt"

# val_results, predictions = result(
#     model=model, 
#     cfg=cfg, 
#     ckpt_path=ckpt_path,
#     model_name="msfv2_scalehead_moe_k4_soft_sum2"
# )

In [ ]:
# import os
# import csv
# import numpy as np
# import torch

# from configs import get_default_cfg
# from model.models.msf2_model_scalehead_moe import MSFv2, MSFGv2

# cfg = get_default_cfg()

# device = cfg["device"]
# in_shape = (cfg["input_len"], 1, 32, 32)

# model = MSFv2(
# image_size=cfg["image_size"],
# in_chans=1,
# tin=cfg["input_len"],
# tout=cfg["pred_len"],
# d_model=cfg["d_model"],
# depth=cfg["depth"],
# num_heads=cfg["num_heads"],
# mlp_ratio=cfg["mlp_ratio"],
# dropout=cfg["dropout"],
# attn_dropout=cfg["attn_dropout"],
# spatial_scales=cfg["spatial_scales"],
# fusion=cfg["fusion"],
# use_pad=cfg["use_pad"],
# use_pos_emb=cfg["use_pos_emb"],
# time_pool=cfg["time_pool"],

# head_num_experts=cfg["head_num_experts"],
# head_top_k=cfg["head_top_k"],
# head_temperature=cfg["head_temperature"],
# aux_balance_coef=cfg["aux_balance_coef"],
# aux_z_coef=cfg["aux_z_coef"],
# ).to(device)


    
# ckpt_path = "checkpoints/msfv2_scalehead_moe_soft_04.pt"

# val_results, predictions = result(
#     model=model, 
#     cfg=cfg, 
#     ckpt_path=ckpt_path,
#     model_name="msfv2_scalehead_moe_soft_04"
# )

In [ ]:
# import os
# import csv
# import numpy as np
# import torch

# from configs import get_default_cfg
# from model.models.msf2_model_scalehead_moe import MSFv2, MSFGv2

# cfg = get_default_cfg()

# device = cfg["device"]
# in_shape = (cfg["input_len"], 1, 32, 32)

# model = MSFv2(
# image_size=cfg["image_size"],
# in_chans=1,
# tin=cfg["input_len"],
# tout=cfg["pred_len"],
# d_model=cfg["d_model"],
# depth=cfg["depth"],
# num_heads=cfg["num_heads"],
# mlp_ratio=cfg["mlp_ratio"],
# dropout=cfg["dropout"],
# attn_dropout=cfg["attn_dropout"],
# spatial_scales=cfg["spatial_scales"],
# fusion=cfg["fusion"],
# use_pad=cfg["use_pad"],
# use_pos_emb=cfg["use_pos_emb"],
# time_pool=cfg["time_pool"],

# head_num_experts=cfg["head_num_experts"],
# head_top_k=cfg["head_top_k"],
# head_temperature=cfg["head_temperature"],
# aux_balance_coef=cfg["aux_balance_coef"],
# aux_z_coef=cfg["aux_z_coef"],
# ).to(device)


    
# ckpt_path = "checkpoints/msfv2_scalehead_moe_top1.pt"

# val_results, predictions = result(
#     model=model, 
#     cfg=cfg, 
#     ckpt_path=ckpt_path,
#     model_name="MSFv2_scalehead_moe_top1"
# )

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# # 파일 로드
# mask = np.load('./model_comparison/MSFv2_preds.npy')  # 실제 mask 파일 경로로 수정


# # 실제 True 개수 확인
# print(f"Total points: {mask.shape}")
# print(f"True (Active) points: {np.sum(mask)}")
# print(f"False (Masked) points: {mask.size - np.sum(mask)}")